#Nama Anggota Kelompok :
- Ravi Arnan Irianto (2305551076)
- Richad Krishnadana Primantara (2305551151)
- Putu Satria Arya Putra (2305551122)

# Tugas STKI: Perbandingan TF-IDF dan VSM

Notebook ini membandingkan dua metode perankingan dokumen:
- **TF-IDF**: penjumlahan skor bobot term query pada setiap dokumen.
- **VSM**: cosine similarity antara vektor query dan vektor dokumen berbasis TF-IDF.

Semua komentar, label, dan output menggunakan Bahasa Indonesia.

## Install Library

Library inti sudah diinstal pada cell pertama menggunakan `PySastrawi`.

In [ ]:
!pip install PySastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 1.6 MB/s eta 0:00:00


## Import & Konfigurasi

In [ ]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Import dan konfigurasi selesai. Teks tabel tidak dipotong.')

Import dan konfigurasi selesai. Teks tabel tidak dipotong.


## Dokumen Korpus

Korpus berisi 4 dokumen yang bersumber dari jurnal:

**"Analisis Term Frequency Inverse Document Frequency (TF-IDF) dalam Temu Kembali Informasi pada Dokumen Teks"**
*(Dwi Septiani & Ica Isabela, SINTESIA: Jurnal Sistem dan Teknologi Informasi Indonesia, Vol. 01 No. 2, Maret 2022)*

Setiap dokumen merepresentasikan satu bagian utama dari jurnal:
- **D1** -> Pendahuluan
- **D2** -> Metode Penelitian
- **D3** -> Hasil dan Pembahasan
- **D4** -> Penutup (Kesimpulan & Saran)

In [ ]:
dokumen = {
    'D1': 'Sistem temu kembali informasi pada dokumen teks telah mengalami '
          'perkembangan signifikan dalam representasi dokumen dan pencocokan '
          'query. Representasi dokumen kini menggunakan pemodelan vektor dalam '
          'ruang multidimensi dengan teknik seperti Word2Vec GloVe dan BERT '
          'untuk memahami konteks dan hubungan antar kata. Algoritma pencocokan '
          'query terus ditingkatkan dengan mempertimbangkan konteks sinonim '
          'dan relasi antar kata menggunakan model seperti BM25 dan '
          'Transformer-based models. Sistem temu kembali informasi bertujuan '
          'menghasilkan dokumen paling relevan berdasarkan keyword query '
          'pengguna dengan tiga elemen utama yaitu masukan pemroses dan '
          'keluaran. Penelitian ini menggunakan metode Term Frequency Inverse '
          'Document Frequency TF-IDF sebagai teknik pengolahan teks dan '
          'pemodelan bahasa alami.',

    'D2': 'Metode Term Frequency Inverse Document Frequency TF-IDF adalah '
          'teknik pengolahan teks dan pemodelan bahasa alami yang bertujuan '
          'mengevaluasi seberapa penting suatu kata dalam sebuah dokumen dalam '
          'konteks koleksi dokumen yang lebih besar. Metode ini memperhitungkan '
          'dua faktor utama yaitu Term Frequency TF yang mengukur seberapa '
          'sering suatu kata muncul dalam dokumen dan Inverse Document '
          'Frequency IDF yang mengukur seberapa penting suatu kata dalam '
          'konteks keseluruhan koleksi dokumen. Nilai TF dihitung dari jumlah '
          'kemunculan term dibagi jumlah total kata dalam dokumen sedangkan '
          'IDF dihitung dengan logaritma dari pembagian jumlah total dokumen '
          'dengan jumlah dokumen yang mengandung term tersebut. Bobot akhir '
          'TF-IDF diperoleh dari perkalian nilai TF dan IDF yang mencerminkan '
          'tingkat pentingnya suatu term dalam dokumen dibandingkan seluruh '
          'koleksi dokumen.',

    'D3': 'Sistem temu kembali informasi adalah sistem komputer yang dirancang '
          'untuk mencari dan mengorganisir informasi relevan dari koleksi '
          'dokumen berdasarkan permintaan pengguna melalui komponen masukan '
          'pemroses dan keluaran. Metode TF-IDF berkontribusi signifikan dalam '
          'pencarian dokumen teks dengan menghasilkan representasi numerik yang '
          'menggambarkan pentingnya term berdasarkan frekuensi kemunculannya '
          'dan inversi frekuensinya dalam koleksi dokumen. Pembuatan indeks '
          'TF-IDF melibatkan tahapan pra-pemrosesan dokumen perhitungan TF '
          'perhitungan IDF perkalian TF-IDF hingga pembangunan indeks yang '
          'memungkinkan akses cepat ke dokumen relevan. Metode TF-IDF memiliki '
          'keunggulan dalam memberikan bobot tepat pada term menangani term '
          'umum dan skalabilitas pada koleksi besar namun memiliki keterbatasan '
          'dalam memperhatikan konteks makna panjang dokumen relasi antar term '
          'serta sinonim dan polisemi.',

    'D4': 'Sistem temu kembali informasi adalah sistem komputer yang bertujuan '
          'memberikan respons akurat terhadap kebutuhan informasi pengguna '
          'melalui komponen masukan query pemroses dan keluaran dokumen relevan. '
          'Metode TF-IDF sebagai metode statistik menggabungkan skor Term '
          'Frequency dan Inverse Document Frequency untuk mengukur pentingnya '
          'term dalam dokumen yang digunakan dalam pemeringkatan pencarian '
          'informasi relevan pemfilteran dokumen dan ekstraksi ringkasan. '
          'Kelebihan metode TF-IDF meliputi representasi term yang sederhana '
          'dan skalabilitas namun kelemahannya mencakup ketidakmampuan '
          'memperhatikan urutan kata dan hubungan semantik antar term. '
          'Penelitian selanjutnya disarankan mengembangkan sistem temu kembali '
          'informasi dengan mengintegrasikan metode TF-IDF bersama kecerdasan '
          'buatan dan analisis sentimen untuk meningkatkan kinerja dalam '
          'menangani big data dan pencarian bahasa alami.'
}

label_dokumen = {
    'D1': 'Pendahuluan',
    'D2': 'Metode Penelitian',
    'D3': 'Hasil dan Pembahasan',
    'D4': 'Penutup'
}

df_korpus = pd.DataFrame({
    'Dokumen': list(dokumen.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in dokumen.keys()],
    'Teks Asli': list(dokumen.values())
})
display(df_korpus)

,Dokumen,Bagian Jurnal,Teks Asli
0,D1,Pendahuluan,Sistem temu kembali informasi pada dokumen teks telah mengalami perkembangan signifikan dalam representasi dokumen dan pencocokan query. Representasi dokumen kini menggunakan pemodelan vektor dalam ruang multidimensi dengan teknik seperti Word2Vec GloVe dan BERT untuk memahami konteks dan hubungan antar kata. Algoritma pencocokan query terus ditingkatkan dengan mempertimbangkan konteks sinonim dan relasi antar kata menggunakan model seperti BM25 dan Transformer-based models. Sistem temu kembali informasi bertujuan menghasilkan dokumen paling relevan berdasarkan keyword query pengguna dengan tiga elemen utama yaitu masukan pemroses dan keluaran. Penelitian ini menggunakan metode Term Frequency Inverse Document Frequency TF-IDF sebagai teknik pengolahan teks dan pemodelan bahasa alami.
1,D2,Metode Penelitian,Metode Term Frequency Inverse Document Frequency TF-IDF adalah teknik pengolahan teks dan pemodelan bahasa alami yang bertujuan mengevaluasi seberapa penting suatu kata dalam sebuah dokumen dalam konteks koleksi dokumen yang lebih besar. Metode ini memperhitungkan dua faktor utama yaitu Term Frequency TF yang mengukur seberapa sering suatu kata muncul dalam dokumen dan Inverse Document Frequency IDF yang mengukur seberapa penting suatu kata dalam konteks keseluruhan koleksi dokumen. Nilai TF dihitung dari jumlah kemunculan term dibagi jumlah total kata dalam dokumen sedangkan IDF dihitung dengan logaritma dari pembagian jumlah total dokumen dengan jumlah dokumen yang mengandung term tersebut. Bobot akhir TF-IDF diperoleh dari perkalian nilai TF dan IDF yang mencerminkan tingkat pentingnya suatu term dalam dokumen dibandingkan seluruh koleksi dokumen.
2,D3,Hasil dan Pembahasan,Sistem temu kembali informasi adalah sistem komputer yang dirancang untuk mencari dan mengorganisir informasi relevan dari koleksi dokumen berdasarkan permintaan pengguna melalui komponen masukan pemroses dan keluaran. Metode TF-IDF berkontribusi signifikan dalam pencarian dokumen teks dengan menghasilkan representasi numerik yang menggambarkan pentingnya term berdasarkan frekuensi kemunculannya dan inversi frekuensinya dalam koleksi dokumen. Pembuatan indeks TF-IDF melibatkan tahapan pra-pemrosesan dokumen perhitungan TF perhitungan IDF perkalian TF-IDF hingga pembangunan indeks yang memungkinkan akses cepat ke dokumen relevan. Metode TF-IDF memiliki keunggulan dalam memberikan bobot tepat pada term menangani term umum dan skalabilitas pada koleksi besar namun memiliki keterbatasan dalam memperhatikan konteks makna panjang dokumen relasi antar term serta sinonim dan polisemi.
3,D4,Penutup,Sistem temu kembali informasi adalah sistem komputer yang bertujuan memberikan respons akurat terhadap kebutuhan informasi pengguna melalui komponen masukan query pemroses dan keluaran dokumen relevan. Metode TF-IDF sebagai metode statistik menggabungkan skor Term Frequency dan Inverse Document Frequency untuk mengukur pentingnya term dalam dokumen yang digunakan dalam pemeringkatan pencarian informasi relevan pemfilteran dokumen dan ekstraksi ringkasan. Kelebihan metode TF-IDF meliputi representasi term yang sederhana dan skalabilitas namun kelemahannya mencakup ketidakmampuan memperhatikan urutan kata dan hubungan semantik antar term. Penelitian selanjutnya disarankan mengembangkan sistem temu kembali informasi dengan mengintegrasikan metode TF-IDF bersama kecerdasan buatan dan analisis sentimen untuk meningkatkan kinerja dalam menangani big data dan pencarian bahasa alami.


## Preprocessing

Pipeline `preprocess(text)`:
1. Case folding
2. Cleaning (regex: hapus angka dan tanda baca)
3. Tokenizing
4. Stopword Removal (`StopWordRemoverFactory`)
5. Stemming (`StemmerFactory`)

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    teks_tanpa_stopword = stopword_remover.remove(' '.join(tokens))
    tokens_tanpa_stopword = teks_tanpa_stopword.split()
    tokens_stem = [stemmer.stem(token) for token in tokens_tanpa_stopword]
    return tokens_stem

hasil_preproses = {nama_dok: preprocess(teks) for nama_dok, teks in dokumen.items()}

df_preproses = pd.DataFrame({
    'Dokumen': list(hasil_preproses.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in hasil_preproses.keys()],
    'Token Hasil Preprocessing': [' '.join(v) for v in hasil_preproses.values()]
})
display(df_preproses)

,Dokumen,Bagian Jurnal,Token Hasil Preprocessing
0,D1,Pendahuluan,sistem temu informasi dokumen teks alami kembang signifikan representasi dokumen cocok query representasi dokumen model vektor ruang multidimensi teknik word vec glove bert paham konteks hubung algoritma cocok query tingkat timbang konteks sinonim relasi model bm transformer based models sistem temu informasi tuju hasil dokumen relevan dasar keyword query guna elemen utama masuk pemroses keluar teliti metode term frequency inverse document frequency tf idf teknik olah teks model bahasa alami
1,D2,Metode Penelitian,metode term frequency inverse document frequency tf idf teknik olah teks model bahasa alami tuju evaluasi dokumen konteks koleksi dokumen metode hitung faktor utama term frequency tf ukur muncul dokumen inverse document frequency idf ukur konteks koleksi dokumen nilai tf hitung muncul term bagi total dokumen idf hitung logaritma bagi total dokumen dokumen kandung term bobot tf idf oleh kali nilai tf idf cermin tingkat term dokumen banding koleksi dokumen
2,D3,Hasil dan Pembahasan,sistem temu informasi sistem komputer rancang cari organisir informasi relevan koleksi dokumen dasar minta guna komponen masuk pemroses keluar metode tf idf kontribusi signifikan cari dokumen teks hasil representasi numerik gambar term dasar frekuensi muncul inversi frekuensi koleksi dokumen buat indeks tf idf libat tahap pra pemrosesan dokumen hitung tf hitung idf kali tf idf bangun indeks akses cepat dokumen relevan metode tf idf milik unggul bobot term tangan term umum skalabilitas koleksi milik batas perhati konteks makna dokumen relasi term sinonim polisemi
3,D4,Penutup,sistem temu informasi sistem komputer tuju respons akurat butuh informasi guna komponen masuk query pemroses keluar dokumen relevan metode tf idf metode statistik gabung skor term frequency inverse document frequency ukur term dokumen peringkat cari informasi relevan filter dokumen ekstraksi ringkas lebih metode tf idf liput representasi term sederhana skalabilitas lemah cakup ketidakmampuan perhati urut hubung semantik term teliti saran kembang sistem temu informasi integrasi metode tf idf cerdas buat analisis sentimen tingkat kerja tangan big data cari bahasa alami


## Perhitungan TF-IDF Manual



In [ ]:
nama_dokumen = list(hasil_preproses.keys())
N = len(nama_dokumen)

vocab = sorted({term for tokens in hasil_preproses.values() for term in tokens})

# TF manual
tf = {}
for dok, tokens in hasil_preproses.items():
    counter = Counter(tokens)
    total_token = len(tokens) if len(tokens) > 0 else 1
    tf[dok] = {term: counter.get(term, 0) / total_token for term in vocab}

# DF manual
df_freq = {term: sum(1 for dok in nama_dokumen if tf[dok][term] > 0) for term in vocab}

# IDF manual
idf = {term: math.log10(N / df_freq[term]) if df_freq[term] > 0 else 0.0 for term in vocab}

# TF-IDF manual
tfidf = {
    dok: {term: tf[dok][term] * idf[term] for term in vocab}
    for dok in nama_dokumen
}

print(f'Jumlah vocabulary (term unik): {len(vocab)} term')
print(f'Jumlah dokumen: {N}')

Jumlah vocabulary (term unik): 124 term
Jumlah dokumen: 4


## Bobot TF-IDF per Paragraf (Dokumen)



In [ ]:
top_k = 5

print('BOBOT TF-IDF PER PARAGRAF (DOKUMEN)')
print('=' * 65)

bobot_data = []

for dok in nama_dokumen:
    skor_term = {t: tfidf[dok][t] for t in vocab if tfidf[dok][t] > 0}
    top_terms = sorted(skor_term.items(), key=lambda x: x[1], reverse=True)[:top_k]

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 65)
    for rank, (term, skor) in enumerate(top_terms, 1):
        bar = '█' * int(skor * 600)
        print(f'  {rank}. {term:<22} TF-IDF: {skor:.6f}  {bar}')

    keyword_str = ', '.join([t for t, _ in top_terms])
    bobot_data.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Keyword Utama (Top-5)': keyword_str,
        'Skor TF-IDF Tertinggi': round(top_terms[0][1], 6) if top_terms else 0
    })

print('\n')
print('=' * 65)
print('TABEL RINGKASAN KEYWORD UTAMA PER PARAGRAF')
print('=' * 65)
df_bobot = pd.DataFrame(bobot_data)
display(df_bobot)

BOBOT TF-IDF PER PARAGRAF (DOKUMEN)

D1 — Pendahuluan
-----------------------------------------------------------------
  1. cocok                  TF-IDF: 0.017202  ██████████
  2. model                  TF-IDF: 0.012901  ███████
  3. query                  TF-IDF: 0.012901  ███████
  4. algoritma              TF-IDF: 0.008601  █████
  5. based                  TF-IDF: 0.008601  █████

D2 — Metode Penelitian
-----------------------------------------------------------------
  1. bagi                   TF-IDF: 0.017202  ██████████
  2. nilai                  TF-IDF: 0.017202  ██████████
  3. total                  TF-IDF: 0.017202  ██████████
  4. hitung                 TF-IDF: 0.012901  ███████
  5. koleksi                TF-IDF: 0.012901  ███████

D3 — Hasil dan Pembahasan
-----------------------------------------------------------------
  1. frekuensi              TF-IDF: 0.014507  ████████
  2. indeks                 TF-IDF: 0.014507  ████████
  3. milik                  TF-IDF: 0.0

,Dokumen,Bagian Jurnal,Keyword Utama (Top-5),Skor TF-IDF Tertinggi
0,D1,Pendahuluan,"cocok, model, query, algoritma, based",0.017202
1,D2,Metode Penelitian,"bagi, nilai, total, hitung, koleksi",0.017202
2,D3,Hasil dan Pembahasan,"frekuensi, indeks, milik, koleksi, akses",0.014507
3,D4,Penutup,"akurat, analisis, big, butuh, cakup",0.007526


## Ranking dengan TF-IDF



In [ ]:
query = 'sistem temu kembali informasi metode term frequency inverse document frequency'
query_tokens = preprocess(query)
query_terms = [term for term in query_tokens if term in vocab]

skor_tfidf = {}
for dok in nama_dokumen:
    skor_tfidf[dok] = sum(tfidf[dok][term] for term in query_terms)

ranking_tfidf = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor TF-IDF': [skor_tfidf[d] for d in nama_dokumen]
}).sort_values('Skor TF-IDF', ascending=False).reset_index(drop=True)
ranking_tfidf['Ranking TF-IDF'] = ranking_tfidf.index + 1

print('Query:', query)
print('Term query setelah preprocessing:', query_terms)
display(ranking_tfidf)

Query: sistem temu kembali informasi metode term frequency inverse document frequency
Term query setelah preprocessing: ['sistem', 'temu', 'informasi', 'metode', 'term', 'frequency', 'inverse', 'document', 'frequency']


,Dokumen,Bagian Jurnal,Skor TF-IDF,Ranking TF-IDF
0,D4,Penutup,0.023426,1
1,D1,Pendahuluan,0.021418,2
2,D2,Metode Penelitian,0.021418,3
3,D3,Hasil dan Pembahasan,0.007526,4


## Ranking dengan VSM (Cosine Similarity)



In [ ]:
def cosine_similarity_manual(vec_a, vec_b):
    pembilang = float(np.dot(vec_a, vec_b))
    penyebut = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return pembilang / penyebut if penyebut != 0 else 0.0

counter_query = Counter(query_terms)
total_query = len(query_terms) if len(query_terms) > 0 else 1
tf_query = {term: counter_query.get(term, 0) / total_query for term in vocab}
vektor_query = np.array([tf_query[term] * idf[term] for term in vocab], dtype=float)

skor_vsm = {}
for dok in nama_dokumen:
    vektor_dok = np.array([tfidf[dok][term] for term in vocab], dtype=float)
    skor_vsm[dok] = cosine_similarity_manual(vektor_query, vektor_dok)

ranking_vsm = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor VSM': [skor_vsm[d] for d in nama_dokumen]
}).sort_values('Skor VSM', ascending=False).reset_index(drop=True)
ranking_vsm['Ranking VSM'] = ranking_vsm.index + 1

display(ranking_vsm)

,Dokumen,Bagian Jurnal,Skor VSM,Ranking VSM
0,D4,Penutup,0.183949,1
1,D2,Metode Penelitian,0.155703,2
2,D1,Pendahuluan,0.152036,3
3,D3,Hasil dan Pembahasan,0.054441,4


## Perbandingan Hasil Ranking



In [ ]:
hasil_perbandingan = (
    ranking_tfidf[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF']]
    .merge(ranking_vsm[['Dokumen', 'Skor VSM', 'Ranking VSM']], on='Dokumen', how='inner')
)
hasil_perbandingan = hasil_perbandingan.sort_values('Ranking TF-IDF').reset_index(drop=True)

display(hasil_perbandingan[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF', 'Skor VSM', 'Ranking VSM']])

,Dokumen,Bagian Jurnal,Skor TF-IDF,Ranking TF-IDF,Skor VSM,Ranking VSM
0,D4,Penutup,0.023426,1,0.183949,1
1,D1,Pendahuluan,0.021418,2,0.152036,3
2,D2,Metode Penelitian,0.021418,3,0.155703,2
3,D3,Hasil dan Pembahasan,0.007526,4,0.054441,4


##Sumarization

In [ ]:
def skor_kalimat(kalimat, tfidf_dok, vocab, idf):
    """Hitung skor kalimat = rata-rata TF-IDF semua token dalam kalimat."""
    tokens = preprocess(kalimat)
    if not tokens:
        return 0.0
    total_skor = 0.0
    for token in tokens:
        if token in idf:
            tf_token = tokens.count(token) / len(tokens)
            total_skor += tf_token * idf[token]
    return total_skor / len(tokens)


print('SUMMARIZATION JURNAL BERBASIS TF-IDF')
print('=' * 70)
print(f'Judul Jurnal : Analisis TF-IDF dalam Temu Kembali Informasi pada Dokumen Teks')
print(f'Penulis      : Dwi Septiani & Ica Isabela')
print(f'Query        : {query}')
print('=' * 70)

summary_rows = []
kalimat_summary = []

for dok in nama_dokumen:
    teks_asli = dokumen[dok]

    # Pecah dokumen menjadi kalimat berdasarkan tanda titik
    kalimat_list = [k.strip() for k in teks_asli.split('.') if len(k.strip()) > 20]

    # Hitung skor setiap kalimat
    skor_per_kalimat = []
    for kal in kalimat_list:
        skor = skor_kalimat(kal, tfidf[dok], vocab, idf)
        skor_per_kalimat.append((kal, skor))

    # Ambil kalimat dengan skor tertinggi
    kalimat_terbaik, skor_terbaik = max(skor_per_kalimat, key=lambda x: x[1])

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 70)
    print('Skor semua kalimat:')
    for i, (kal, skor) in enumerate(skor_per_kalimat, 1):
        tanda = ' ◄ TERPILIH' if kal == kalimat_terbaik else ''
        print(f'  [{i}] (skor: {skor:.5f}){tanda}')
        print(f'       "{kal[:80]}..."' if len(kal) > 80 else f'       "{kal}"')
    print(f'\n  >>> Kalimat Terpilih (skor: {skor_terbaik:.5f}):')
    print(f'      "{kalimat_terbaik}."')

    kalimat_summary.append(kalimat_terbaik + '.')
    summary_rows.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Jumlah Kalimat': len(kalimat_list),
        'Skor Kalimat Terpilih': round(skor_terbaik, 5),
        'Kalimat Summary': kalimat_terbaik + '.'
    })

# Tampilkan tabel summary
print('\n')
print('=' * 70)
print('TABEL HASIL SUMMARIZATION PER DOKUMEN')
print('=' * 70)
df_summary_result = pd.DataFrame(summary_rows)
display(df_summary_result)

# Tampilkan ringkasan akhir
print('\n')
print('=' * 70)
print('RINGKASAN AKHIR JURNAL (EXTRACTIVE SUMMARIZATION)')
print('=' * 70)
ringkasan_akhir = ' '.join(kalimat_summary)
print(ringkasan_akhir)

SUMMARIZATION JURNAL BERBASIS TF-IDF
Judul Jurnal : Analisis TF-IDF dalam Temu Kembali Informasi pada Dokumen Teks
Penulis      : Dwi Septiani & Ica Isabela
Query        : sistem temu kembali informasi metode term frequency inverse document frequency

D1 — Pendahuluan
----------------------------------------------------------------------
Skor semua kalimat:
  [1] (skor: 0.01566)
       "Sistem temu kembali informasi pada dokumen teks telah mengalami perkembangan sig..."
  [2] (skor: 0.03046)
       "Representasi dokumen kini menggunakan pemodelan vektor dalam ruang multidimensi ..."
  [3] (skor: 0.03354) ◄ TERPILIH
       "Algoritma pencocokan query terus ditingkatkan dengan mempertimbangkan konteks si..."
  [4] (skor: 0.01380)
       "Sistem temu kembali informasi bertujuan menghasilkan dokumen paling relevan berd..."
  [5] (skor: 0.01035)
       "Penelitian ini menggunakan metode Term Frequency Inverse Document Frequency TF-I..."

  >>> Kalimat Terpilih (skor: 0.03354):
      "Algori

,Dokumen,Bagian Jurnal,Jumlah Kalimat,Skor Kalimat Terpilih,Kalimat Summary
0,D1,Pendahuluan,5,0.03354,Algoritma pencocokan query terus ditingkatkan dengan mempertimbangkan konteks sinonim dan relasi antar kata menggunakan model seperti BM25 dan Transformer-based models.
1,D2,Metode Penelitian,4,0.02812,Nilai TF dihitung dari jumlah kemunculan term dibagi jumlah total kata dalam dokumen sedangkan IDF dihitung dengan logaritma dari pembagian jumlah total dokumen dengan jumlah dokumen yang mengandung term tersebut.
2,D3,Hasil dan Pembahasan,4,0.01767,Pembuatan indeks TF-IDF melibatkan tahapan pra-pemrosesan dokumen perhitungan TF perhitungan IDF perkalian TF-IDF hingga pembangunan indeks yang memungkinkan akses cepat ke dokumen relevan.
3,D4,Penutup,4,0.02022,Kelebihan metode TF-IDF meliputi representasi term yang sederhana dan skalabilitas namun kelemahannya mencakup ketidakmampuan memperhatikan urutan kata dan hubungan semantik antar term.




RINGKASAN AKHIR JURNAL (EXTRACTIVE SUMMARIZATION)
Algoritma pencocokan query terus ditingkatkan dengan mempertimbangkan konteks sinonim dan relasi antar kata menggunakan model seperti BM25 dan Transformer-based models. Nilai TF dihitung dari jumlah kemunculan term dibagi jumlah total kata dalam dokumen sedangkan IDF dihitung dengan logaritma dari pembagian jumlah total dokumen dengan jumlah dokumen yang mengandung term tersebut. Pembuatan indeks TF-IDF melibatkan tahapan pra-pemrosesan dokumen perhitungan TF perhitungan IDF perkalian TF-IDF hingga pembangunan indeks yang memungkinkan akses cepat ke dokumen relevan. Kelebihan metode TF-IDF meliputi representasi term yang sederhana dan skalabilitas namun kelemahannya mencakup ketidakmampuan memperhatikan urutan kata dan hubungan semantik antar term.
